# RI-JK UHF Hessian：CP-HF 分解 (3) Krylov 求解器

本文档对应 `02-6-decomp_cphf_3.ipynb` 的 UHF 版本。重点：

1. UHF 的 CP-HF 求解空间维度为 `[nset, nmo*nocc_α + nmo*nocc_β]`，两个自旋通道**展平拼接**为单个向量后送入 Krylov 求解器。
2. 对比 PySCF 的 `lib.krylov` 与外部实现 `krylov_block` 在 UHF 下的等价性。
3. 由于 `krylov_block` 是 RHF/UHF 通用的（输入只要求 2D `[nset, n]`），不需要做任何修改即可用于 UHF。

In [1]:
from pyscf import gto, scf, lib, df, hessian
from pyscf.hessian import uhf as uhf_hess
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper
from krylov_block import krylov_block

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", charge=2, spin=2, max_memory=32000).build()

In [3]:
mf = scf.UHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_u_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_u_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_u_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
α, β = 0, 1

mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao = mo_coeff.shape[1]
nmo = mo_coeff.shape[2]

mocc = [mo_coeff[x][:, mo_occ[x] > 0] for x in (α, β)]
mvir = [mo_coeff[x][:, mo_occ[x] == 0] for x in (α, β)]
nocc = [mocc[x].shape[1] for x in (α, β)]
nvir = [mvir[x].shape[1] for x in (α, β)]
eocc = [mo_energy[x][mo_occ[x] > 0] for x in (α, β)]
evir = [mo_energy[x][mo_occ[x] == 0] for x in (α, β)]

natm = mol.natm
aoslices = mol.aoslice_by_atom()

n_a = nmo * nocc[α]
n_b = nmo * nocc[β]
nov = n_a + n_b
print(f"nov (= nmo*nocc_α + nmo*nocc_β): {n_a} + {n_b} = {nov}")

nov (= nmo*nocc_α + nmo*nocc_β): 245 + 147 = 392


In [6]:
def ovlp_deriv1_generator(mol):
    int1e_ipovlp = mol.intor("int1e_ipovlp")

    def get_ovlp_deriv_at_atoms(A):
        shl0, shl1, p0, p1 = aoslices[A]
        s1ao = np.zeros((3, nao, nao))
        s1ao[:, p0:p1, :] += - int1e_ipovlp[:, p0:p1]
        s1ao[:, :, p0:p1] += - int1e_ipovlp[:, p0:p1].transpose(0, 2, 1)
        return s1ao
    return get_ovlp_deriv_at_atoms

## Problem-Setting

构造和 `05-5` 中相同的 vind_vo 算子与 mo1_base，但这次我们要用它去比较不同的 Krylov 求解器。

In [7]:
f1ao = mf_hess.make_h1(mo_coeff, mo_occ)
f1ao = [np.asarray(f1ao[x]).reshape(-1, nao, nao) for x in (α, β)]
s1ao = np.array([ovlp_deriv1_generator(mol)(A) for A in range(natm)]).reshape(-1, nao, nao)

e_ai = [1.0 / (evir[x][:, None] - eocc[x][None, :]) for x in (α, β)]

f1mo = [mo_coeff[x].T @ f1ao[x] @ mocc[x] for x in (α, β)]
s1mo = [mo_coeff[x].T @ s1ao @ mocc[x] for x in (α, β)]
hsmo = [f1mo[x] - s1mo[x] * eocc[x] for x in (α, β)]

mo1_base = [np.zeros_like(hsmo[x]) for x in (α, β)]
for x in (α, β):
    mo1_base[x][:, nocc[x]:] = -hsmo[x][:, nocc[x]:] * e_ai[x]
    mo1_base[x][:, :nocc[x]] = -s1mo[x][:, :nocc[x]] * 0.5

def pack_uhf(arr_list):
    n = arr_list[α].shape[0]
    return np.hstack([arr_list[α].reshape(n, -1), arr_list[β].reshape(n, -1)])

def unpack_uhf(flat):
    n = flat.shape[0]
    a = flat[:, :n_a].reshape(n, nmo, nocc[α])
    b = flat[:, n_a:].reshape(n, nmo, nocc[β])
    return [a, b]

mo1_base_flat = pack_uhf(mo1_base)

fvind = uhf_hess.gen_vind(mf, mo_coeff, mo_occ)

def vind_vo(mo1_flat):
    mo1_flat = mo1_flat.reshape(-1, nov)
    v_flat = fvind(mo1_flat).reshape(-1, nov)
    v_list = unpack_uhf(v_flat)
    for x in (α, β):
        v_list[x][:, nocc[x]:, :] *= e_ai[x]
        v_list[x][:, :nocc[x], :] = 0
    return pack_uhf(v_list)

print("mo1_base_flat shape:", mo1_base_flat.shape)

mo1_base_flat shape: (12, 392)


## 比较不同求解器

在 RHF 下 `02-6` 比较过 `lib.krylov`、`krylov_glm`（自写 GMRES）、`krylov_block`（自写块 Krylov）。UHF 下我们重点比较 `lib.krylov` 与 `krylov_block`，确认两者结果一致；GMRES 实现细节不影响数学结论，可参考 RHF 文档。

**关键观察**：UHF 与 RHF 的唯一差异只在 `vind_vo` 这个算子本身（接受 `[nset, nov]` 而非 `[nset, nmo*nocc]`）。Krylov 算法本体完全相同。

In [8]:
mo1_pyscf = lib.krylov(vind_vo, mo1_base_flat, tol=1e-8, verbose=5)
print("mo1_pyscf shape:", mo1_pyscf.shape)

krylov cycle 0  r = 0.0845608


krylov cycle 1  r = 0.0195907


krylov cycle 2  r = 0.00292991


krylov cycle 3  r = 0.000355158


krylov cycle 4  r = 3.38347e-05


krylov cycle 5  r = 2.78232e-06


krylov cycle 6  r = 2.21142e-07


mo1_pyscf shape: (12, 392)


In [9]:
mo1_block = krylov_block(vind_vo, mo1_base_flat, tol=1e-8)
print("mo1_block shape:", mo1_block.shape)
print("krylov_block matches lib.krylov:", np.allclose(mo1_block, mo1_pyscf, atol=1e-6, rtol=1e-4))

mo1_block shape: (12, 392)
krylov_block matches lib.krylov: True


## 与 PySCF `solve_mo1` 结果交叉核验

解出 `mo1` 后做收尾，把 vir 块用完整的 `hsmo + vind` 重算，并提取 `mo_e1`，与 `mf_hess.solve_mo1` 的结果对照。

In [10]:
mo1_block_list = unpack_uhf(mo1_block)
for x in (α, β):
    mo1_block_list[x][:, :nocc[x]] = mo1_base[x][:, :nocc[x]]

v_final = unpack_uhf(fvind(pack_uhf(mo1_block_list)))
mo_e1_block = [None, None]
for x in (α, β):
    hsmo_full = f1mo[x] - s1mo[x] * eocc[x] + v_final[x]
    mo1_block_list[x][:, nocc[x]:] = hsmo_full[:, nocc[x]:] / (eocc[x] - evir[x][:, None])
    mo_e1_block[x] = hsmo_full[:, :nocc[x]] + mo1_block_list[x][:, :nocc[x]] * (eocc[x][:, None] - eocc[x])

# krylov_block 求解得到的是未 bra-transform 的 U_{p i}^{A, σ}
mo1_block_list = [mo1_block_list[x].reshape(natm, 3, nmo, nocc[x]) for x in (α, β)]
mo_e1_block = [mo_e1_block[x].reshape(natm, 3, nocc[x], nocc[x]) for x in (α, β)]

# 参考：PySCF 的 mf_hess.solve_mo1（返回 bra-transformed 版本）
mo1_bra_ref, mo_e1_ref = mf_hess.solve_mo1(mo_energy, mo_coeff, mo_occ,
                                             mf_hess.make_h1(mo_coeff, mo_occ))
mo1_bra_ref = [np.asarray(mo1_bra_ref[x]) for x in (α, β)]
mo_e1_ref = [np.asarray(mo_e1_ref[x]) for x in (α, β)]

for x in (α, β):
    mo1_btr = np.einsum("pq, Atqi -> Atpi", mo_coeff[x], mo1_block_list[x])
    print(f"spin {x}: krylov_block mo1 (after bra-trans) matches PySCF mo1_bra:", np.allclose(mo1_btr, mo1_bra_ref[x]))
    print(f"spin {x}: krylov_block mo_e1 matches PySCF:", np.allclose(mo_e1_block[x], mo_e1_ref[x]))

spin 0: krylov_block mo1 (after bra-trans) matches PySCF mo1_bra: True
spin 0: krylov_block mo_e1 matches PySCF: True
spin 1: krylov_block mo1 (after bra-trans) matches PySCF mo1_bra: True
spin 1: krylov_block mo_e1 matches PySCF: True


## 小结

- 由于 `krylov_block` 把输入视为 `[nset, n]` 的通用 2D 数组，UHF（n = nmo*nocc_α + nmo*nocc_β）与 RHF（n = nmo*nocc）共享同一份求解器实现，无需任何修改。
- UHF 与 RHF 在数据布局上的唯一差异封装在 `vind_vo` 算子（以及外部 `pack_uhf` / `unpack_uhf` 辅助函数）中。
- 这一观察对应到最终 pyhessref 中：可以在 `RHessSCF` 之外引入 `UHessSCF`，求解器/Krylov 子模块直接复用；不同之处主要在 `compute_dimensionless_cphf_rhs` / `response_mo` / `finalize_cphf` 这几个需要按自旋分通道操作的接口。